In [4]:
!python -m pip install --upgrade pip

In [5]:
!pip --version

pip 26.1.1 from C:\Users\anand\PycharmProjects\Python\JupyterProject1\.venv\Lib\site-packages\pip (python 3.11)



In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import chi2_contingency, norm, ttest_ind
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
import os

warnings.filterwarnings('ignore')
np.random.seed(42)
OUTPUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print("  A/B TESTING FRAMEWORK — STATISTICAL RIGOR")
print("  E-Commerce Checkout Page Optimization")
print("=" * 60)


# ─────────────────────────────────────────────
# 1. SYNTHETIC DATA GENERATION
# ─────────────────────────────────────────────
print("\n[1/8] Generating realistic A/B test data...")

N_CONTROL   = 4720   # users who saw old page
N_TREATMENT = 4918   # users who saw new page

# True conversion rates (ground truth for simulation)
TRUE_CONTROL_CR   = 0.1180   # 11.8% baseline
TRUE_TREATMENT_CR = 0.1270   # 12.7% new page (+0.9 pp lift)

np.random.seed(42)
control_conversions   = np.random.binomial(1, TRUE_CONTROL_CR,   N_CONTROL)
treatment_conversions = np.random.binomial(1, TRUE_TREATMENT_CR, N_TREATMENT)

# Build DataFrame
control_df = pd.DataFrame({
    'user_id':    range(1, N_CONTROL + 1),
    'group':      'control',
    'converted':  control_conversions,
    'revenue':    control_conversions * np.random.lognormal(3.5, 0.6, N_CONTROL)
})
treatment_df = pd.DataFrame({
    'user_id':    range(N_CONTROL + 1, N_CONTROL + N_TREATMENT + 1),
    'group':      'treatment',
    'converted':  treatment_conversions,
    'revenue':    treatment_conversions * np.random.lognormal(3.6, 0.55, N_TREATMENT)
})
df = pd.concat([control_df, treatment_df], ignore_index=True)

print(f"  Total users: {len(df):,}")
print(f"  Control   : {N_CONTROL:,} users")
print(f"  Treatment : {N_TREATMENT:,} users")


# ─────────────────────────────────────────────
# 2. EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────
print("\n[2/8] Exploratory Data Analysis...")

summary = df.groupby('group').agg(
    users       = ('user_id', 'count'),
    conversions = ('converted', 'sum'),
    conv_rate   = ('converted', 'mean'),
    total_rev   = ('revenue', 'sum'),
    avg_rev_user= ('revenue', 'mean'),
    avg_rev_conv= ('revenue', lambda x: x[x > 0].mean())
).reset_index()

summary['conv_rate_pct'] = (summary['conv_rate'] * 100).round(2)
summary['lift_vs_control'] = ((summary['conv_rate'] / summary.loc[summary['group']=='control','conv_rate'].values[0]) - 1) * 100

print(f"\n  {'Group':<12} {'Users':>7} {'Conversions':>12} {'Conv Rate':>10} {'Avg Rev/User':>13}")
print("  " + "-"*58)
for _, row in summary.iterrows():
    print(f"  {row['group']:<12} {int(row['users']):>7,} {int(row['conversions']):>12,} "
          f"{row['conv_rate_pct']:>9.2f}% {row['avg_rev_user']:>12.2f}")


# ─────────────────────────────────────────────
# 3. SAMPLE SIZE & POWER ANALYSIS
# ─────────────────────────────────────────────
print("\n[3/8] Sample Size & Power Analysis...")

alpha   = 0.05
power   = 0.80
p1      = TRUE_CONTROL_CR
p2      = TRUE_TREATMENT_CR
p_pool  = (p1 + p2) / 2
effect  = abs(p2 - p1) / np.sqrt(p_pool * (1 - p_pool))  # Cohen's h approx

z_alpha = norm.ppf(1 - alpha / 2)
z_beta  = norm.ppf(power)
n_required = int(np.ceil(((z_alpha + z_beta) / effect) ** 2))

print(f"  Baseline conversion rate  : {p1*100:.1f}%")
print(f"  Expected new rate         : {p2*100:.1f}%")
print(f"  Minimum detectable effect : {(p2-p1)*100:.1f} percentage points")
print(f"  Required sample (per group): {n_required:,}")
print(f"  Actual sample  (per group): {min(N_CONTROL, N_TREATMENT):,}")
if min(N_CONTROL, N_TREATMENT) >= n_required:
    print(f"  ✓ Sample size is SUFFICIENT for this test")
else:
    print(f"  ✗ Sample size is INSUFFICIENT — results may be unreliable")


# ─────────────────────────────────────────────
# 4. FREQUENTIST STATISTICAL TESTS
# ─────────────────────────────────────────────
print("\n[4/8] Running Statistical Tests...")

ctrl  = df[df['group'] == 'control']
treat = df[df['group'] == 'treatment']

conv_ctrl  = int(ctrl['converted'].sum())
conv_treat = int(treat['converted'].sum())
n_ctrl     = len(ctrl)
n_treat    = len(treat)
cr_ctrl    = conv_ctrl / n_ctrl
cr_treat   = conv_treat / n_treat
lift_abs   = cr_treat - cr_ctrl
lift_rel   = (lift_abs / cr_ctrl) * 100

# Chi-Square Test
contingency = np.array([[conv_ctrl, n_ctrl - conv_ctrl],
                         [conv_treat, n_treat - conv_treat]])
chi2, p_chi2, dof, expected = chi2_contingency(contingency)

# Z-Test for proportions
p_pool_obs = (conv_ctrl + conv_treat) / (n_ctrl + n_treat)
se = np.sqrt(p_pool_obs * (1 - p_pool_obs) * (1/n_ctrl + 1/n_treat))
z_stat = lift_abs / se
p_ztest = 2 * (1 - norm.cdf(abs(z_stat)))

# T-Test on revenue
t_stat, p_ttest = ttest_ind(ctrl['revenue'], treat['revenue'])

# Confidence Interval for lift
se_lift = np.sqrt((cr_ctrl*(1-cr_ctrl)/n_ctrl) + (cr_treat*(1-cr_treat)/n_treat))
ci_low  = lift_abs - z_alpha * se_lift
ci_high = lift_abs + z_alpha * se_lift

print(f"\n  ── Conversion Rate Results ──")
print(f"  Control   CR : {cr_ctrl*100:.3f}%")
print(f"  Treatment CR : {cr_treat*100:.3f}%")
print(f"  Absolute lift: {lift_abs*100:+.3f} pp")
print(f"  Relative lift: {lift_rel:+.2f}%")
print(f"  95% CI for lift: [{ci_low*100:.3f}%, {ci_high*100:.3f}%]")

print(f"\n  ── Chi-Square Test ──")
print(f"  χ² = {chi2:.4f}, p-value = {p_chi2:.4f}, df = {dof}")
print(f"  Result: {'✓ SIGNIFICANT' if p_chi2 < alpha else '✗ NOT SIGNIFICANT'} (α={alpha})")

print(f"\n  ── Z-Test for Proportions ──")
print(f"  Z = {z_stat:.4f}, p-value = {p_ztest:.4f}")
print(f"  Result: {'✓ SIGNIFICANT' if p_ztest < alpha else '✗ NOT SIGNIFICANT'} (α={alpha})")

print(f"\n  ── T-Test on Revenue per User ──")
print(f"  T = {t_stat:.4f}, p-value = {p_ttest:.4f}")
print(f"  Control avg revenue  : ₹{ctrl['revenue'].mean():.2f}")
print(f"  Treatment avg revenue: ₹{treat['revenue'].mean():.2f}")
print(f"  Result: {'✓ SIGNIFICANT' if p_ttest < alpha else '✗ NOT SIGNIFICANT'} (α={alpha})")


# ─────────────────────────────────────────────
# 5. EFFECT SIZE — Cohen's h
# ─────────────────────────────────────────────
print("\n[5/8] Effect Size (Cohen's h)...")

cohens_h = 2 * (np.arcsin(np.sqrt(cr_treat)) - np.arcsin(np.sqrt(cr_ctrl)))
if abs(cohens_h) < 0.2:
    effect_label = "Small"
elif abs(cohens_h) < 0.5:
    effect_label = "Medium"
else:
    effect_label = "Large"

print(f"  Cohen's h = {cohens_h:.4f} → {effect_label} effect")
print("  (Small < 0.2, Medium 0.2-0.5, Large > 0.5)")


# ─────────────────────────────────────────────
# 6. BAYESIAN A/B ANALYSIS
# ─────────────────────────────────────────────
print("\n[6/8] Bayesian A/B Analysis...")

# Beta distribution posterior (uniform prior: Beta(1,1))
alpha_ctrl  = conv_ctrl + 1
beta_ctrl   = n_ctrl - conv_ctrl + 1
alpha_treat = conv_treat + 1
beta_treat  = n_treat - conv_treat + 1

# Monte Carlo simulation
n_samples = 100_000
samples_ctrl  = np.random.beta(alpha_ctrl,  beta_ctrl,  n_samples)
samples_treat = np.random.beta(alpha_treat, beta_treat, n_samples)

prob_treat_better = (samples_treat > samples_ctrl).mean()
expected_lift     = (samples_treat - samples_ctrl).mean()
credible_low      = np.percentile(samples_treat - samples_ctrl, 2.5)
credible_high     = np.percentile(samples_treat - samples_ctrl, 97.5)

print(f"  P(Treatment > Control)      : {prob_treat_better*100:.2f}%")
print(f"  Expected lift (Bayesian)    : {expected_lift*100:+.3f} pp")
print(f"  95% Credible Interval       : [{credible_low*100:.3f}%, {credible_high*100:.3f}%]")
print(f"  Bayesian Decision           : {'✓ DEPLOY Treatment' if prob_treat_better > 0.95 else '⚠ Needs more data'}")


# ─────────────────────────────────────────────
# 7. VISUALIZATIONS
# ─────────────────────────────────────────────
print("\n[7/8] Creating Visualizations...")

# Color palette
C_CTRL  = '#4A90D9'
C_TREAT = '#E8643A'
C_BG    = '#F8F9FA'
C_LINE  = '#2C3E50'
C_GREEN = '#27AE60'
C_RED   = '#E74C3C'

fig = plt.figure(figsize=(18, 20), facecolor='white')
fig.suptitle('A/B Testing Framework — E-Commerce Checkout Page Optimization\nStatistical Rigor Analysis Report',
             fontsize=16, fontweight='bold', y=0.98, color=C_LINE)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)

# ── Plot 1: Conversion Rate Comparison ──
ax1 = fig.add_subplot(gs[0, 0])
groups = ['Control (A)\nOld Page', 'Treatment (B)\nNew Page']
rates  = [cr_ctrl * 100, cr_treat * 100]
colors = [C_CTRL, C_TREAT]
bars = ax1.bar(groups, rates, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, rate in zip(bars, rates):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{rate:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
ci_ctrl_err  = z_alpha * np.sqrt(cr_ctrl*(1-cr_ctrl)/n_ctrl) * 100
ci_treat_err = z_alpha * np.sqrt(cr_treat*(1-cr_treat)/n_treat) * 100
ax1.errorbar([0, 1], rates, yerr=[ci_ctrl_err, ci_treat_err],
             fmt='none', color='black', capsize=6, linewidth=1.5)
ax1.set_title('Conversion Rate\n(with 95% CI)', fontweight='bold', fontsize=11, color=C_LINE)
ax1.set_ylabel('Conversion Rate (%)', fontsize=10)
ax1.set_ylim(0, max(rates) * 1.25)
ax1.set_facecolor(C_BG)
ax1.spines[['top','right']].set_visible(False)
ax1.tick_params(axis='x', labelsize=9)

# ── Plot 2: Lift Distribution (Bayesian posterior) ──
ax2 = fig.add_subplot(gs[0, 1])
lift_samples = (samples_treat - samples_ctrl) * 100
ax2.hist(lift_samples, bins=80, color=C_TREAT, alpha=0.7, edgecolor='white', linewidth=0.3)
ax2.axvline(0, color=C_RED, linestyle='--', linewidth=2, label='No effect')
ax2.axvline(expected_lift*100, color=C_GREEN, linestyle='-', linewidth=2,
            label=f'Expected lift: {expected_lift*100:+.2f}pp')
ax2.axvline(credible_low*100, color='gray', linestyle=':', linewidth=1.5, label='95% Credible Interval')
ax2.axvline(credible_high*100, color='gray', linestyle=':', linewidth=1.5)
ax2.fill_betweenx([0, ax2.get_ylim()[1] if ax2.get_ylim()[1] > 0 else 1000],
                  credible_low*100, credible_high*100, alpha=0.08, color=C_GREEN)
ax2.set_title('Bayesian Posterior\nLift Distribution', fontweight='bold', fontsize=11, color=C_LINE)
ax2.set_xlabel('Lift (percentage points)', fontsize=10)
ax2.set_ylabel('Frequency', fontsize=10)
ax2.legend(fontsize=8)
ax2.set_facecolor(C_BG)
ax2.spines[['top','right']].set_visible(False)

# ── Plot 3: P-value Significance Gauge ──
ax3 = fig.add_subplot(gs[0, 2])
tests  = ['Chi-Square', 'Z-Test', 'T-Test\n(Revenue)']
pvals  = [p_chi2, p_ztest, p_ttest]
pcolors = [C_GREEN if p < alpha else C_RED for p in pvals]
bars3 = ax3.barh(tests, pvals, color=pcolors, height=0.5, edgecolor='white')
ax3.axvline(alpha, color=C_RED, linestyle='--', linewidth=2, label=f'α = {alpha}')
for bar, p in zip(bars3, pvals):
    ax3.text(max(p + 0.002, 0.002), bar.get_y() + bar.get_height()/2,
             f'p={p:.4f}', va='center', fontsize=10, fontweight='bold',
             color=C_GREEN if p < alpha else C_RED)
ax3.set_title('P-values Across\nAll Tests', fontweight='bold', fontsize=11, color=C_LINE)
ax3.set_xlabel('p-value', fontsize=10)
ax3.legend(fontsize=9)
ax3.set_facecolor(C_BG)
ax3.spines[['top','right']].set_visible(False)
ax3.set_xlim(0, max(pvals) * 1.5)

# ── Plot 4: Revenue Distribution ──
ax4 = fig.add_subplot(gs[1, :2])
rev_ctrl_conv  = ctrl[ctrl['revenue'] > 0]['revenue']
rev_treat_conv = treat[treat['revenue'] > 0]['revenue']
bins = np.linspace(0, max(rev_ctrl_conv.max(), rev_treat_conv.max()), 50)
ax4.hist(rev_ctrl_conv, bins=bins, alpha=0.6, color=C_CTRL,
         label=f'Control (mean ₹{rev_ctrl_conv.mean():.0f})', edgecolor='white')
ax4.hist(rev_treat_conv, bins=bins, alpha=0.6, color=C_TREAT,
         label=f'Treatment (mean ₹{rev_treat_conv.mean():.0f})', edgecolor='white')
ax4.axvline(rev_ctrl_conv.mean(),  color=C_CTRL,  linestyle='--', linewidth=2)
ax4.axvline(rev_treat_conv.mean(), color=C_TREAT, linestyle='--', linewidth=2)
ax4.set_title('Revenue Distribution (Converted Users Only)', fontweight='bold', fontsize=11, color=C_LINE)
ax4.set_xlabel('Revenue per Transaction (₹)', fontsize=10)
ax4.set_ylabel('Number of Users', fontsize=10)
ax4.legend(fontsize=10)
ax4.set_facecolor(C_BG)
ax4.spines[['top','right']].set_visible(False)

# ── Plot 5: Power Analysis Curve ──
ax5 = fig.add_subplot(gs[1, 2])
sample_sizes = np.arange(500, 10000, 100)
powers_curve = []
for n in sample_sizes:
    se_n = np.sqrt(p_pool_obs * (1 - p_pool_obs) * 2 / n)
    z_n  = abs(lift_abs) / se_n
    pw   = 1 - norm.cdf(z_alpha - z_n) + norm.cdf(-z_alpha - z_n)
    powers_curve.append(pw)
ax5.plot(sample_sizes, powers_curve, color=C_TREAT, linewidth=2)
ax5.axhline(0.80, color=C_GREEN, linestyle='--', linewidth=1.5, label='80% power threshold')
ax5.axvline(n_required, color=C_RED, linestyle=':', linewidth=1.5,
            label=f'Required n={n_required:,}')
ax5.fill_between(sample_sizes, 0.80, powers_curve,
                 where=[p >= 0.80 for p in powers_curve], alpha=0.15, color=C_GREEN)
ax5.set_title('Statistical Power\nvs Sample Size', fontweight='bold', fontsize=11, color=C_LINE)
ax5.set_xlabel('Sample Size (per group)', fontsize=10)
ax5.set_ylabel('Statistical Power', fontsize=10)
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)
ax5.set_facecolor(C_BG)
ax5.spines[['top','right']].set_visible(False)

# ── Plot 6: Confidence Interval Plot ──
ax6 = fig.add_subplot(gs[2, 0])
x_range = np.linspace(-0.03, 0.06, 300)
se_diff  = np.sqrt(cr_ctrl*(1-cr_ctrl)/n_ctrl + cr_treat*(1-cr_treat)/n_treat)
pdf_vals = norm.pdf(x_range, lift_abs, se_diff)
ax6.plot(x_range * 100, pdf_vals, color=C_TREAT, linewidth=2)
ax6.fill_between(x_range * 100, pdf_vals,
                 where=(x_range >= ci_low) & (x_range <= ci_high),
                 alpha=0.3, color=C_TREAT, label='95% CI')
ax6.axvline(0, color=C_RED, linestyle='--', linewidth=2, label='Null (no effect)')
ax6.axvline(lift_abs * 100, color=C_GREEN, linewidth=2, label=f'Observed lift: {lift_abs*100:+.2f}pp')
ax6.set_title('Sampling Distribution\nof Lift', fontweight='bold', fontsize=11, color=C_LINE)
ax6.set_xlabel('Lift (percentage points)', fontsize=10)
ax6.set_ylabel('Probability Density', fontsize=10)
ax6.legend(fontsize=8)
ax6.set_facecolor(C_BG)
ax6.spines[['top','right']].set_visible(False)

# ── Plot 7: Funnel — Conversions ──
ax7 = fig.add_subplot(gs[2, 1])
funnel_labels = ['Total Users', 'Engaged\n(>30s session)', 'Added to Cart', 'Purchased']
funnel_ctrl   = [n_ctrl, int(n_ctrl*0.62), int(n_ctrl*0.24), conv_ctrl]
funnel_treat  = [n_treat, int(n_treat*0.67), int(n_treat*0.27), conv_treat]
x = np.arange(len(funnel_labels))
w = 0.35
ax7.bar(x - w/2, funnel_ctrl,  width=w, label='Control',   color=C_CTRL,  alpha=0.85)
ax7.bar(x + w/2, funnel_treat, width=w, label='Treatment', color=C_TREAT, alpha=0.85)
ax7.set_xticks(x)
ax7.set_xticklabels(funnel_labels, fontsize=9)
ax7.set_title('Conversion Funnel\nComparison', fontweight='bold', fontsize=11, color=C_LINE)
ax7.set_ylabel('Users', fontsize=10)
ax7.legend(fontsize=9)
ax7.set_facecolor(C_BG)
ax7.spines[['top','right']].set_visible(False)

# ── Plot 8: Summary Scorecard ──
ax8 = fig.add_subplot(gs[2, 2])
ax8.axis('off')
scorecard = [
    ("Metric",               "Control",          "Treatment",        "Verdict"),
    ("Conv Rate",            f"{cr_ctrl*100:.2f}%", f"{cr_treat*100:.2f}%", f"{lift_abs*100:+.2f}pp"),
    ("Avg Revenue/User",     f"₹{ctrl['revenue'].mean():.0f}",
                             f"₹{treat['revenue'].mean():.0f}",
                             "Treatment ↑"),
    ("Chi-Square p",         "-",                f"{p_chi2:.4f}",    "✓ Sig." if p_chi2 < alpha else "✗"),
    ("Z-Test p",             "-",                f"{p_ztest:.4f}",   "✓ Sig." if p_ztest < alpha else "✗"),
    ("Bayesian P(B>A)",      "-",                f"{prob_treat_better*100:.1f}%", "✓ Strong" if prob_treat_better > 0.95 else "Weak"),
    ("Cohen's h",            "-",                f"{cohens_h:.4f}",  effect_label),
    ("Sample Sufficient?",   f"Need {n_required:,}", f"Have {min(N_CONTROL,N_TREATMENT):,}", "✓ Yes" if min(N_CONTROL,N_TREATMENT)>=n_required else "✗ No"),
]
table = ax8.table(cellText=scorecard[1:], colLabels=scorecard[0],
                  cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(8.5)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor('#DDDDDD')
    if r == 0:
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#F0F4F8')
    else:
        cell.set_facecolor('white')
    if c == 3 and r > 0:
        txt = cell.get_text().get_text()
        if '✓' in txt or '↑' in txt:
            cell.set_facecolor('#D5F5E3')
        elif '✗' in txt:
            cell.set_facecolor('#FADBD8')
ax8.set_title('Summary Scorecard', fontweight='bold', fontsize=11, color=C_LINE, pad=10)

plt.savefig(f"{OUTPUT_DIR}/ab_test_full_analysis.png", dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close()
print("  ✓ Main analysis chart saved")


# ─────────────────────────────────────────────
# PLOT 2: Bayesian Beta Distributions
# ─────────────────────────────────────────────
fig2, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='white')
fig2.suptitle('Bayesian Posterior Beta Distributions', fontsize=14, fontweight='bold', color=C_LINE)

x_pdf = np.linspace(0.08, 0.17, 500)
pdf_ctrl  = stats.beta.pdf(x_pdf, alpha_ctrl, beta_ctrl)
pdf_treat = stats.beta.pdf(x_pdf, alpha_treat, beta_treat)

ax = axes[0]
ax.plot(x_pdf * 100, pdf_ctrl,  color=C_CTRL,  linewidth=2.5, label=f'Control CR ~ Beta({alpha_ctrl},{beta_ctrl})')
ax.plot(x_pdf * 100, pdf_treat, color=C_TREAT, linewidth=2.5, label=f'Treatment CR ~ Beta({alpha_treat},{beta_treat})')
ax.fill_between(x_pdf * 100, pdf_ctrl,  alpha=0.2, color=C_CTRL)
ax.fill_between(x_pdf * 100, pdf_treat, alpha=0.2, color=C_TREAT)
ax.axvline(cr_ctrl*100,  color=C_CTRL,  linestyle='--', linewidth=1.5)
ax.axvline(cr_treat*100, color=C_TREAT, linestyle='--', linewidth=1.5)
ax.set_title('Posterior Distributions of Conversion Rate', fontweight='bold', color=C_LINE)
ax.set_xlabel('Conversion Rate (%)', fontsize=10)
ax.set_ylabel('Probability Density', fontsize=10)
ax.legend(fontsize=9)
ax.set_facecolor(C_BG)
ax.spines[['top','right']].set_visible(False)

ax2b = axes[1]
lift_pct = lift_samples
ax2b.hist(lift_pct, bins=100, color=C_TREAT, alpha=0.75, edgecolor='white', linewidth=0.2, density=True)
ax2b.axvline(0, color=C_RED, linestyle='--', linewidth=2, label='No effect (H₀)')
ax2b.axvline(np.percentile(lift_pct, 2.5),  color='gray', linestyle=':', linewidth=1.5)
ax2b.axvline(np.percentile(lift_pct, 97.5), color='gray', linestyle=':', linewidth=1.5, label='95% Credible Interval')
ax2b.axvline(lift_pct.mean(), color=C_GREEN, linewidth=2,
             label=f'Mean lift: {lift_pct.mean():+.2f}pp')
x_fill = np.linspace(np.percentile(lift_pct, 2.5), np.percentile(lift_pct, 97.5), 200)
from scipy.stats import gaussian_kde
kde = gaussian_kde(lift_pct)
ax2b.fill_between(x_fill, kde(x_fill), alpha=0.15, color=C_GREEN)
prob_pos = (lift_pct > 0).mean()
ax2b.set_title(f'P(Treatment > Control) = {prob_pos*100:.2f}%', fontweight='bold', color=C_LINE)
ax2b.set_xlabel('Lift in Conversion Rate (pp)', fontsize=10)
ax2b.set_ylabel('Density', fontsize=10)
ax2b.legend(fontsize=9)
ax2b.set_facecolor(C_BG)
ax2b.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ab_test_bayesian.png", dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close()
print("  ✓ Bayesian analysis chart saved")


# ─────────────────────────────────────────────
# 8. BUSINESS RECOMMENDATION REPORT
# ─────────────────────────────────────────────
print("\n[8/8] Business Recommendation Report")
print("=" * 60)

decision = "DEPLOY TREATMENT (New Checkout Page)" if (p_chi2 < alpha and prob_treat_better > 0.95) else "DO NOT DEPLOY — Insufficient Evidence"
est_extra_conv = int(lift_abs * 100000)
est_revenue    = est_extra_conv * treat[treat['revenue']>0]['revenue'].mean()

print(f"""
  ┌─────────────────────────────────────────────────┐
  │         EXECUTIVE SUMMARY — A/B TEST RESULT     │
  └─────────────────────────────────────────────────┘

  TEST PERIOD   : 30 days (simulated)
  TEST GROUPS   : {N_CONTROL:,} control | {N_TREATMENT:,} treatment users
  HYPOTHESIS    : New checkout page increases conversion rate

  ── KEY FINDINGS ──────────────────────────────────
  • Conversion lift    : {lift_abs*100:+.3f} pp ({lift_rel:+.2f}% relative)
  • 95% Confidence Int : [{ci_low*100:.3f}%, {ci_high*100:.3f}%]
  • Chi-Square p-value : {p_chi2:.4f} {'← SIGNIFICANT ✓' if p_chi2 < alpha else '← NOT SIGNIFICANT ✗'}
  • Z-Test p-value     : {p_ztest:.4f} {'← SIGNIFICANT ✓' if p_ztest < alpha else '← NOT SIGNIFICANT ✗'}
  • Bayesian P(B>A)    : {prob_treat_better*100:.2f}% (threshold: 95%)
  • Effect size        : {cohens_h:.4f} ({effect_label} — Cohen's h)
  • Sample adequacy    : {'✓ Sufficient' if min(N_CONTROL,N_TREATMENT)>=n_required else '✗ Insufficient'}

  ── BUSINESS IMPACT (per 100,000 users) ───────────
  • Additional conversions : ~{est_extra_conv:,}
  • Estimated revenue gain : ₹{est_revenue:,.0f}

  ── DECISION ──────────────────────────────────────
  ➤  {decision}

  ── CAVEATS ───────────────────────────────────────
  • Monitor for novelty effect in first 2 weeks post-launch
  • Run segment analysis (mobile vs desktop, new vs returning)
  • Statistical significance ≠ practical significance — validate ROI
  • Consider long-term metrics (retention, LTV) before full rollout

  ── NEXT STEPS ────────────────────────────────────
  1. Run segmented analysis by device and user cohort
  2. Perform multi-variate test on checkout page elements
  3. Monitor post-launch metrics for 4 weeks
  4. Evaluate impact on customer lifetime value (LTV)
""")

print(f"  All outputs saved to: {OUTPUT_DIR}/")
print("  • ab_test_full_analysis.png")
print("  • ab_test_bayesian.png")




  A/B TESTING FRAMEWORK — STATISTICAL RIGOR
  E-Commerce Checkout Page Optimization

[1/8] Generating realistic A/B test data...
  Total users: 9,638
  Control   : 4,720 users
  Treatment : 4,918 users

[2/8] Exploratory Data Analysis...

  Group          Users  Conversions  Conv Rate  Avg Rev/User
  ----------------------------------------------------------
  control        4,720          533     11.29%         4.44
  treatment      4,918          601     12.22%         5.46

[3/8] Sample Size & Power Analysis...
  Baseline conversion rate  : 11.8%
  Expected new rate         : 12.7%
  Minimum detectable effect : 0.9 percentage points
  Required sample (per group): 10,417
  Actual sample  (per group): 4,720
  ✗ Sample size is INSUFFICIENT — results may be unreliable

[4/8] Running Statistical Tests...

  ── Conversion Rate Results ──
  Control   CR : 11.292%
  Treatment CR : 12.220%
  Absolute lift: +0.928 pp
  Relative lift: +8.22%
  95% CI for lift: [-0.358%, 2.214%]

  ── Chi-Squar